In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../"); sys.path.append("../../utils")

import pandas as pd

from typing import Tuple
from pydantic import create_model
from kb_mapper import chat_json
from mapping_utils import pending, run_chunked

In [ ]:
def translate_offer_to_english(job_title: str, description: str, model: str = "qwen3:8b") -> Tuple[str, str]:
    """Translate a job title + description to English via the LLM.

    Returns ``(eng_occupation, eng_description)``. The occupation is kept short
    (2-8 words) so it embeds cleanly for ESCO occupation retrieval downstream.
    """
    format_output = create_model("TranslationOutput", eng_occupation=(str, ...), eng_description=(str, ...), )
    system_prompt = (
            "You are an expert job-market translator.\n"
            "Translate the input job title and job description to English.\n"
            "Keep eng_occupation concise (2-8 words).\n"
            "Keep meaning faithful to the source text.\n"
            "Return only valid JSON matching the provided schema."
    )
    result = chat_json(
        output_model=format_output,
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user","content": f"Job title: {job_title}\nJob description: {description}",},
        ],
        task_name="translation",
    )
    return result.eng_occupation.strip(), result.eng_description.strip()

In [ ]:
OUT = ("mapping/translated.csv", ["id_j", "raw_title", "eng_title", "raw_description", "eng_description"])

In [ ]:
preprocessed_jobs = pd.read_json("mapping/preprocessed_jobs.json")[["id_j", "job_title", "description"]]
todo = pending(preprocessed_jobs, OUT)

In [ ]:
def translate_row(row):
    eng_title, eng_description = translate_offer_to_english(row["job_title"], row["description"])
    return {
        "id_j": row["id_j"],
        "raw_title": row["job_title"],
        "eng_title": eng_title,
        "raw_description": row["description"],
        "eng_description": eng_description,
    }

run_chunked(todo, translate_row, OUT)